In [1]:
# =============================================================
# Teste do agente de monitoramento
# Pipeline com triagem, deteccao da CNN, planejamento, briefing e ordem tatica.
# =============================================================
import os

from agent.settings import STATIONS_DIR
from agent.vision import classify_image
from agent.monitoring_flow import (
    make_llm, build_recursos, _build_report,
    triar_mensagem, _build_triage_agent,
    _build_planner_agent, _build_alert_agent, _build_navegador_agent,
)
from agent.situation import (
    QUADRO, FROTA, resumo_md, frota_md,
    _alocar_demandas, planejar_demandas, despachar_alocacao,
    gerar_briefing, briefing_da_missao,
)

# Servidor LLM dos agentes (LiteLLM)
PROVIDER = "ollama" # ollama, openai, anthropic, azure, gemini
MODEL = "text-qwen3-4b" # Esse é o modelo salvo no meu PC, vale listar os modelos com "ollama list"
SERVER_URL = "http://localhost:11434"
API_KEY = ""

llm = make_llm(provider=PROVIDER, model=MODEL,
               base_url=SERVER_URL, api_key=API_KEY or None)
QUADRO.limpar()
FROTA.limpar()

In [2]:
# =============================================================
# Triagem das mensagens de vitimas
# =============================================================
mensagens = [
    "socorro tem 4 pessoas presas no telhado da rua das flores 120, a agua ta subindo rapido e tem uma criança pequena",
    "meu pai e cadeirante e a agua ja ta na altura do peito aqui na vila são jose, preciso de resgate urgente agora",
    "minha vó ta sozinha no bairro navegantes e a rua alagou toda, ela nao consegue sair de casa",
    "somos 8 pessoas no abrigo da escola municipal, sem agua potavel nem comida ha 2 dias",
    "alguem sabe se o mercado do centro abriu? queria comprar pão",
]

triador = _build_triage_agent(llm)
for m in mensagens:
    r = triar_mensagem(m, triador)
    print(m)
    if r is None:
        print("  triagem indisponivel (sem LLM)\n")
        continue
    print(f"  local={r.local} | pessoas={r.pessoas} | necessidade={r.necessidade} | urgencia={r.urgencia}")
    print(f"  resumo: {r.resumo}\n")
    QUADRO.publicar_vitima(r)

╭───────────────────────────────────────────── 🤖 LiteAgent Started ──────────────────────────────────────────────╮
│                                                                                                                 │
│  LiteAgent Started                                                                                              │
│  Role: Agente de Triagem de Pedidos de Socorro em Enchentes                                                     │
│                                                                                                                 │
│  Status: In Progress                                                                                            │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── 🤖 LiteAgent Started ──────────────────────────────────────────────╮
│                                                                                                                 │
│  LiteAgent Session Started                                                                                      │
│  Name: Agente de Triagem de Pedidos de Socorro em Enchentes                                                     │
│                                                                                                                 │
│  id: bb12b5b7-de76-4596-ac59-32cd3f7ad856                                                                       │
│  role: Agente de Triagem de Pedidos de Socorro em Enchentes                                                     │
│                                                                                                                 │
│  goal: Ler uma mensagem de um possível afetado e extrair, de forma estruturada e confiável, quem precisa de     │
│  ajuda, onde, quantas pessoas e com qual urgência.                                                              │
│                                                                                                                 │
│  backstory: Você é um operador de central de emergência treinado para ler mensagens caóticas de pessoas em      │
│  pânico (de WhatsApp, SMS ou redes sociais) e transformá-las em registros objetivos e acionáveis. Você nunca    │
│  inventa dados: o que não estiver na mensagem, você marca como não informado.                                   │
│                                                                                                                 │
│  tools: None                                                                                                    │
│  verbose: False                                                                                                 │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────── ✅ LiteAgent Completed ─────────────────────────────────────────────╮
│                                                                                                                 │
│  LiteAgent Completed                                                                                            │
│  Role: Agente de Triagem de Pedidos de Socorro em Enchentes                                                     │
│                                                                                                                 │
│  id: bb12b5b7-de76-4596-ac59-32cd3f7ad856                                                                       │
│  role: Agente de Triagem de Pedidos de Socorro em Enchentes                                                     │
│                                                                                                                 │
│  goal: Ler uma mensagem de um possível afetado e extrair, de forma estruturada e confiável, quem precisa de     │
│  ajuda, onde, quantas pessoas e com qual urgência.                                                              │
│                                                                                                                 │
│  backstory: Você é um operador de central de emergência treinado para ler mensagens caóticas de pessoas em      │
│  pânico (de WhatsApp, SMS ou redes sociais) e transformá-las em registros objetivos e acionáveis. Você nunca    │
│  inventa dados: o que não estiver na mensagem, você marca como não informado.                                   │
│                                                                                                                 │
│  tools: None                                                                                                    │
│  verbose: False                                                                                                 │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

socorro tem 4 pessoas presas no telhado da rua das flores 120, a agua ta subindo rapido e tem uma criança pequena
  local=rua das flores 120 | pessoas=4 | necessidade=resgate | urgencia=alta
  resumo: 4 pessoas presas no telhado da rua das flores 120; água subindo rapidamente; há uma criança pequena.



╭───────────────────────────────────────────── 🤖 LiteAgent Started ──────────────────────────────────────────────╮
│                                                                                                                 │
│  LiteAgent Started                                                                                              │
│  Role: Agente de Triagem de Pedidos de Socorro em Enchentes                                                     │
│                                                                                                                 │
│  Status: In Progress                                                                                            │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

somos 8 pessoas no abrigo da escola municipal, sem agua potavel nem comida ha 2 dias
  local=abrigo da escola municipal | pessoas=8 | necessidade=mantimentos | urgencia=media
  resumo: 8 pessoas no abrigo da escola municipal, sem água potável nem comida há 2 dias.



╭───────────────────────────────────────────── 🤖 LiteAgent Started ──────────────────────────────────────────────╮
│                                                                                                                 │
│  LiteAgent Started                                                                                              │
│  Role: Agente de Triagem de Pedidos de Socorro em Enchentes                                                     │
│                                                                                                                 │
│  Status: In Progress                                                                                            │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── 🤖 LiteAgent Started ──────────────────────────────────────────────╮
│                                                                                                                 │
│  LiteAgent Session Started                                                                                      │
│  Name: Agente de Triagem de Pedidos de Socorro em Enchentes                                                     │
│                                                                                                                 │
│  id: bb12b5b7-de76-4596-ac59-32cd3f7ad856                                                                       │
│  role: Agente de Triagem de Pedidos de Socorro em Enchentes                                                     │
│                                                                                                                 │
│  goal: Ler uma mensagem de um possível afetado e extrair, de forma estruturada e confiável, quem precisa de     │
│  ajuda, onde, quantas pessoas e com qual urgência.                                                              │
│                                                                                                                 │
│  backstory: Você é um operador de central de emergência treinado para ler mensagens caóticas de pessoas em      │
│  pânico (de WhatsApp, SMS ou redes sociais) e transformá-las em registros objetivos e acionáveis. Você nunca    │
│  inventa dados: o que não estiver na mensagem, você marca como não informado.                                   │
│                                                                                                                 │
│  tools: None                                                                                                    │
│  verbose: False                                                                                                 │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────── ✅ LiteAgent Completed ─────────────────────────────────────────────╮
│                                                                                                                 │
│  LiteAgent Completed                                                                                            │
│  Role: Agente de Triagem de Pedidos de Socorro em Enchentes                                                     │
│                                                                                                                 │
│  id: bb12b5b7-de76-4596-ac59-32cd3f7ad856                                                                       │
│  role: Agente de Triagem de Pedidos de Socorro em Enchentes                                                     │
│                                                                                                                 │
│  goal: Ler uma mensagem de um possível afetado e extrair, de forma estruturada e confiável, quem precisa de     │
│  ajuda, onde, quantas pessoas e com qual urgência.                                                              │
│                                                                                                                 │
│  backstory: Você é um operador de central de emergência treinado para ler mensagens caóticas de pessoas em      │
│  pânico (de WhatsApp, SMS ou redes sociais) e transformá-las em registros objetivos e acionáveis. Você nunca    │
│  inventa dados: o que não estiver na mensagem, você marca como não informado.                                   │
│                                                                                                                 │
│  tools: None                                                                                                    │
│  verbose: False                                                                                                 │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

alguem sabe se o mercado do centro abriu? queria comprar pão
  local=mercado do centro | pessoas=1 | necessidade=mantimentos | urgencia=baixa
  resumo: Uma pessoa quer comprar pão no mercado do centro.



In [ ]:
# =============================================================
# Drone: a CNN classifica imagens da estacao e publica eventos
# =============================================================
estacao = "Centro_Historico"
pasta = os.path.join(STATIONS_DIR, estacao)
for nome in sorted(os.listdir(pasta)):
    res = classify_image(os.path.join(pasta, nome))
    rep = _build_report(estacao.replace("_", " "), [res])
    print(f"{nome}: prob={res.probability:.1%} | enchente={res.flooded} | severidade={rep.severity}")
    if res.flooded:
        QUADRO.publicar_deteccao(rep.station, res.probability, rep.severity)

In [ ]:
# =============================================================
# Quadro de situacao consolidado (drones + triagem)
# =============================================================
print(resumo_md(QUADRO))

In [ ]:
# =============================================================
# Planejador: aloca a frota sobre as demandas e despacha
# =============================================================
FROTA.definir_total(build_recursos(Helicóptero=1, Bote=2, Equipe_terrestre=3))
demandas = QUADRO.demandas()
aloc = _alocar_demandas(demandas, FROTA.disponiveis_lista())
plano = planejar_demandas(aloc, _build_planner_agent(llm))
despachar_alocacao(aloc, FROTA)
print(plano)

In [ ]:
# =============================================================
# Monitoramento: briefing da situacao
# =============================================================
print(gerar_briefing(QUADRO.demandas(), _build_alert_agent(llm)))

In [ ]:
# =============================================================
# Navegador: ordem tatica de uma missao despachada
# =============================================================
m = FROTA.ativas[0]
print(f"{m.recurso} -> {m.local} ({m.tipo}, prioridade {m.prioridade})")
print(briefing_da_missao(m, _build_navegador_agent(llm)))

In [ ]:
# =============================================================
# Estado final da frota
# =============================================================
print(frota_md(FROTA))